Look for any matches with incomplete data.

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
DATA_PROCESSED = ROOT / "data_processed"

in_path = DATA_PROCESSED / "03_tiebreak_pressure_flags.parquet"
df = pd.read_parquet(in_path)

In [5]:
match_counts = df["match_id"].value_counts()
print("Total unique matches:", match_counts.shape[0])
match_counts.head(10)

Total unique matches: 335


match_id
2012-frenchopen-2408    261
2021-frenchopen-2601    244
2013-wimbledon-2502     239
2021-frenchopen-2503    234
2012-wimbledon-2501     233
2024-wimbledon-2602     231
2014-frenchopen-2701    228
2019-ausopen-2701       228
2017-wimbledon-2504     227
2013-wimbledon-2601     226
Name: count, dtype: int64

First check if there are any games without 'is_match_end' = True

In [6]:
match_end_counts = (
    df.groupby("match_id")["is_match_end"]
      .any()         
)
bad_matches = match_end_counts[~match_end_counts].index

print("Matches without match end:", len(bad_matches))

df = df[~df["match_id"].isin(bad_matches)].copy()

print("Remaining matches:", df["match_id"].nunique())
print("Remaining rows:", len(df))

Matches without match end: 0
Remaining matches: 335
Remaining rows: 48094


Check for any retirements.

In [7]:
import numpy as np
import pandas as pd

df = df.sort_values(["match_id","SetNo","GameNo","PointNumber"]).copy()

def add_sets_won_upto(m: pd.DataFrame) -> pd.DataFrame:
    m = m.copy()

    # NA-safe set_end
    m["is_set_end"] = m["is_set_end"].fillna(False).astype(bool)

    # set final games (max within set)
    set_final = (
        m.groupby("SetNo")[["P1GamesWon","P2GamesWon"]]
         .max()
         .sort_index()
    )

    # set winner: 1 / 2 / 0 (0 = unknown/invalid set)
    set_winner = np.where(
        set_final["P1GamesWon"] > set_final["P2GamesWon"], 1,
        np.where(set_final["P2GamesWon"] > set_final["P1GamesWon"], 2, 0)
    )
    set_winner = pd.Series(set_winner, index=set_final.index).astype("int64")

    m["set_winner"] = m["SetNo"].map(set_winner).fillna(0).astype("int64")

    # only count the set once (on is_set_end row)
    p1_set_completed = ((m["set_winner"] == 1) & (m["is_set_end"])).fillna(False).astype("int64")
    p2_set_completed = ((m["set_winner"] == 2) & (m["is_set_end"])).fillna(False).astype("int64")

    m["P1SetsWon_upto"] = p1_set_completed.cumsum().astype("int64")
    m["P2SetsWon_upto"] = p2_set_completed.cumsum().astype("int64")

    return m

df = df.groupby("match_id", group_keys=False).apply(add_sets_won_upto)


/var/folders/sk/00rkx0hd027cbgk6d5rxrl7w0000gn/T/ipykernel_32388/3743578000.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("match_id", group_keys=False).apply(add_sets_won_upto)


In [8]:
last = (
    df.sort_values(["match_id","SetNo","GameNo","PointNumber"])
      .groupby("match_id")
      .tail(1)
)

print(last[["P1SetsWon_upto","P2SetsWon_upto"]].value_counts())


P1SetsWon_upto  P2SetsWon_upto
2               0                 97
0               2                 90
                0                 56
2               1                 49
1               2                 42
                1                  1
Name: count, dtype: int64


In [9]:

bad_matches = (
    df[df["is_match_end"]]
      .loc[(df["P1SetsWon_upto"] == 0) & (df["P2SetsWon_upto"] == 0), "match_id"]
      .unique()
)
print("Bad matches:", len(bad_matches))


check = (
    df[df["match_id"].isin(bad_matches)]
      .groupby("match_id")["is_set_end"]
      .apply(lambda s: bool(pd.Series(s).fillna(False).any()))
      .value_counts()
)
print(check)

m0 = bad_matches[0]
df[df["match_id"] == m0][["match_id","SetNo","GameNo","PointNumber","is_set_end","is_match_end","P1GamesWon","P2GamesWon"]].tail(30)


Bad matches: 56
is_set_end
True    56
Name: count, dtype: int64


,match_id,SetNo,GameNo,PointNumber,is_set_end,is_match_end,P1GamesWon,P2GamesWon
27217,2018-ausopen-2501,2,13,89,False,False,0,0
27218,2018-ausopen-2501,2,13,90,False,False,0,0
27219,2018-ausopen-2501,2,13,91,False,False,0,0
27220,2018-ausopen-2501,2,13,92,False,False,0,0
27221,2018-ausopen-2501,2,14,93,False,False,0,0
27222,2018-ausopen-2501,2,14,94,False,False,0,0
27223,2018-ausopen-2501,2,14,95,False,False,0,0
27224,2018-ausopen-2501,2,14,96,False,False,0,0
27225,2018-ausopen-2501,2,14,97,False,False,0,0
27226,2018-ausopen-2501,2,15,98,False,False,0,0


In [10]:
completed_matches = (
    df.groupby("match_id")[["P1SetsWon_upto", "P2SetsWon_upto"]]
      .max()
      .query("P1SetsWon_upto == 2 or P2SetsWon_upto == 2")
      .index
)

df = df[df["match_id"].isin(completed_matches)]


Save final dataset

In [11]:
OUT_DIR = DATA_PROCESSED
out_path = OUT_DIR / "04_final_filtered_dataset.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path)
print("Shape:", df.shape)

Saved: /Users/leventezsiga/Documents/Documents - Levente’s MacBook Air/VU/Thesis_P2-P3/tennis-pressure/data_processed/04_final_filtered_dataset.parquet
Shape: (39804, 50)
